# Bronze layer — sổ tay chạy thử
Thăm dò và chạy thử tương tác cho tầng Bronze. **Chỉ đọc** trên các bảng thật (không đụng dữ liệu của job nền). Code sản xuất nằm ở `bronze_stream.py` cùng thư mục.

In [ ]:
# --- bootstrap: cho phép import gtl_session dù notebook nằm ở thư mục con ---
import sys
from pathlib import Path

for c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
          Path.home() / "working/projects/Governed-Transaction-Lakehouse/spark"]:
    if (c / "gtl_session.py").exists():
        sys.path.insert(0, str(c))
        print("spark dir:", c)
        break
else:
    raise RuntimeError("khong tim thay gtl_session.py — chay jupyter tu trong project")

In [ ]:
# Xây SparkSession (lần đầu sẽ kéo JAR từ Maven, hơi lâu).
# Notebook chạy bằng chính venv host: ~/working/gtl-spark-venv/bin/jupyter lab
from gtl_session import get_spark, CATALOG

spark = get_spark("gtl-notebook", master="local[2]", driver_memory="2g")
spark

## 1. Ba bảng Bronze — số dòng

In [ ]:
for t in ["transactions", "accounts", "merchants"]:
    spark.sql(f"REFRESH TABLE {CATALOG}.bronze.{t}")
    n = spark.table(f"{CATALOG}.bronze.{t}").count()
    print(f"{t:13s} {n:>12,d} dòng")

## 2. Phân bố op (c=insert, u=update, d=delete, r=snapshot)
`d` và số dòng value=null (tombstone) phải khớp nhau.

In [ ]:
spark.sql(f"""
    SELECT op, count(*) AS n
    FROM {CATALOG}.bronze.transactions
    GROUP BY op ORDER BY n DESC
""").show()

## 3. Soi một envelope Debezium THÔ
Bronze giữ nguyên cả cục JSON trong cột `value` — đây là bản gốc để audit.

In [ ]:
import json

row = spark.sql(f"""
    SELECT value FROM {CATALOG}.bronze.transactions
    WHERE op = 'u' AND value IS NOT NULL LIMIT 1
""").collect()[0]

env = json.loads(row["value"])
print("op      :", env["op"])
print("before  :", env["before"])
print("after   :", env["after"])
print("amount là string? ->", isinstance(env["after"]["amount"], str),
      "| amount =", env["after"]["amount"])

## 4. Thử parse envelope (tiền thân của việc Silver sẽ làm)
Trích vài trường typed ra từ `value` thô. Silver (Phase 2) sẽ làm đầy đủ + MERGE theo primary key ra current-state.

In [ ]:
from pyspark.sql import functions as F

src = f"{CATALOG}.bronze.transactions"
df = (spark.table(src)
      .where("op IN ('c','u')")
      .select(
          F.get_json_object("value", "$.after.txn_id").cast("bigint").alias("txn_id"),
          F.get_json_object("value", "$.after.status").alias("status"),
          F.get_json_object("value", "$.after.amount").cast("decimal(15,2)").alias("amount"),
          "op", "ingest_ts",
      ))
df.show(5, truncate=False)

## 5. Time-travel — Iceberg giữ lịch sử snapshot
Nền cho audit ở Phase 4: xem bảng đã thay đổi qua từng lần commit.

In [ ]:
spark.sql(f"""
    SELECT committed_at, snapshot_id, operation,
           summary['added-records'] AS added
    FROM {CATALOG}.bronze.transactions.snapshots
    ORDER BY committed_at DESC LIMIT 5
""").show(truncate=False)

## 6. (Tùy chọn) Chạy thử logic streaming — Trigger.AvailableNow
⚠️ **Chỉ chạy khi job nền `bronze_stream.py` đang TẮT** — nếu không sẽ tranh cùng một checkpoint. Cell này rút phần Kafka mới rồi tự dừng (khác với job nền chạy liên tục). Bỏ comment để dùng.

In [ ]:
# from bronze_stream import SOURCES, build_stream, ensure_table, CHECKPOINT_ROOT
# s = SOURCES[0]  # transactions
# ensure_table(spark, s)
# (build_stream(spark, s).writeStream.format("iceberg")
#   .outputMode("append")
#   .option("checkpointLocation", str(CHECKPOINT_ROOT / s["name"]))
#   .trigger(availableNow=True)
#   .toTable(s["table"]).awaitTermination())

---
Xong. `spark.stop()` khi không dùng nữa để trả RAM.

In [ ]:
# spark.stop()